In [1]:
import sys
sys.path.append('..')
from osp import *


In [2]:
with open('../data/feat2egs.json','r') as f:
    feat2egs = json.load(f)
feat2eg = {feat:shorten_eg(egs[0], 25) for feat,egs in feat2egs.items()}
feat2eg

{'deprel_acl': '...ways to XXXEMPHXXX{warrant} our sayin...',
 'deprel_acl:relcl': '..., what is XXXEMPHXXX{more}, could be...',
 'deprel_advcl': '...as being XXXEMPHXXX{argued} and refle...',
 'deprel_advcl:relcl': '...X, that XXXEMPHXXX{settles} the matte...',
 'deprel_advmod': '...ndition is XXXEMPHXXX{not} to be unde...',
 'deprel_amod': '...In this XXXEMPHXXX{important} respect...',
 'deprel_appos': '...whether XXXEMPHXXX{workers} or consum...',
 'deprel_aux': '...d justice XXXEMPHXXX{will} not be exe...',
 'deprel_aux:pass': '...ian Dasein XXXEMPHXXX{was} absorbed i...',
 'deprel_case': '...t our eyes XXXEMPHXXX{to} the green v...',
 'deprel_cc': '...ion to me, XXXEMPHXXX{or} perhaps som...',
 'deprel_cc:preconj': '...bability XXXEMPHXXX{neither} as scien...',
 'deprel_ccomp': '...egin with, XXXEMPHXXX{go} back to the...',
 'deprel_compound': '...skeptical XXXEMPHXXX{self}-understand...',
 'deprel_compound:prt': '...Dewey live XXXEMPHXXX{up} to his phil...',
 'deprel_conj': '...a

In [3]:
df_feats = pd.read_pickle('../data/raw/df_feats3.pkl.gz')
odf = df_feats[['feature','weight','comparison']].groupby(['feature','comparison']).mean(numeric_only=True).reset_index()
odf = odf[odf.comparison.str.contains('Philosophy')]
odf['period1'] = odf.comparison.apply(lambda x: x.split()[0])
odf['period2'] = odf.comparison.apply(lambda x: x.split()[-2])
odf = odf[odf.period1 == odf.period2]
odf = odf.groupby(['period1','feature']).mean(numeric_only=True).reset_index().sort_values('weight', ascending=False)

odf = odf.pivot(index='feature', columns='period1', values='weight')
# odf['feat_diff'] = odf['2000-2025'] - odf['1925-1950']
# odf['feat_diff'] = odf['2000-2025'] - odf['1900-1925']
odf['1900-1950'] = odf[['1900-1925','1925-1950']].mean(axis=1)
odf['1950-2000'] = odf[['1950-1975','2000-2025']].mean(axis=1)
odf['feat_diff'] = odf['2000-2025'] - odf['1900-1950']
# odf['feat_diff'] = odf[['1975-2000','2000-2025']].mean(axis=1) - odf[['1900-1925','1925-1950']].mean(axis=1)
# odf['feat_desc'] = [FEAT2DESC[x]+' ('+x.split('_')[0]+')' for x in odf.index]
odf['feat_desc'] = [FEAT2DESC[x] for x in odf.index]
odf['feat_type'] = [x.split('_')[0] for x in odf.index]
odf['eg'] = odf.index.map(feat2eg).fillna('')
odf = odf.sort_values('feat_diff', ascending=False)
odf['overall'] = odf[['1900-1925','1925-1950','1950-1975','2000-2025']].mean(axis=1)
odf.sort_values('overall', ascending=False)
odf

period1,1900-1925,1925-1950,1950-1975,1975-2000,2000-2025,1900-1950,1950-2000,feat_diff,feat_desc,feat_type,eg,overall
feature,,,,,,,,,,,,
deprel_compound,-1.477498,-1.389497,-0.671225,0.220693,0.011886,-1.433497,-0.329669,1.445383,Compound,deprel,...skeptical XXXEMPHXXX{self}-understand...,-0.881583
pos_NNP,-1.332955,-1.274367,-0.977263,-0.679415,-0.278232,-1.303661,-0.627747,1.025429,"Proper noun, singular",pos,"...Hitler, XXXEMPHXXX{hitler} other crim...",-0.965704
pos_SYM,-0.433158,-0.372921,-0.020470,0.057978,0.263093,-0.403039,0.121312,0.666133,Symbol,pos,...seen shape XXXEMPHXXX{=} rectangulari...,-0.140864
pos_VB,-0.184954,0.081372,0.236791,0.490427,0.566717,-0.051791,0.401754,0.618508,"Verb, base form",pos,...vation to XXXEMPHXXX{imply} that this...,0.174981
deprel_advmod,0.098366,0.064970,0.343020,0.734915,0.599437,0.081668,0.471229,0.517770,Adverbial modifier,deprel,...ndition is XXXEMPHXXX{not} to be unde...,0.276448
...,...,...,...,...,...,...,...,...,...,...,...,...
pos_NNPS,0.199724,-0.400078,-0.292044,-0.376430,-0.575759,-0.100177,-0.433902,-0.475583,"Proper noun, plural",pos,"...e Moral XXXEMPHXXX{sciences}, pp.\nMil...",-0.267039
sent_Cd,0.087987,-0.037246,-0.247193,-0.079422,-0.552662,0.025370,-0.399928,-0.578032,Maximum clause depth,sent,,-0.187279
pos_WP,0.271908,0.075120,0.101625,0.010159,-0.446534,0.173514,-0.172454,-0.620048,Wh-pronoun,pos,"...XXXEMPHXXX{what} is more important, t...",0.000530


In [12]:
xdf = odf.copy()
xdf = xdf[xdf['2000-2025'] > 0.05]
# xdf = xdf[xdf['feat_diff'] > 0.05]
xdf = xdf[['feat_desc', 'eg', '1900-1950', '1950-2000', '2000-2025','feat_diff']]
# xdf = xdf.set_index(['feat_desc','eg','feat_diff']).applymap(lambda x: f'{x:.2f}').reset_index()
for c in ['1900-1950', '1950-2000', '2000-2025']:
    xdf[c] = xdf[c].apply(lambda x: f'{x:.2f}')
xdf['feat_diff'] = xdf['feat_diff'].apply(lambda x: f'{"+" if x > 0 else ""}{x:.2f}')
xdf = xdf.head(25)
xdf = xdf.rename(columns={
    'feat_desc': 'Feature',
    'eg': 'Example',
    'feat_diff': 'Δ(C21--eC20)',
})
xdf

period1,Feature,Example,1900-1950,1950-2000,2000-2025,Δ(C21--eC20)
feature,,,,,,
pos_SYM,Symbol,...seen shape XXXEMPHXXX{=} rectangulari...,-0.40,0.12,0.26,+0.67
pos_VB,"Verb, base form",...vation to XXXEMPHXXX{imply} that this...,-0.05,0.40,0.57,+0.62
deprel_advmod,Adverbial modifier,...ndition is XXXEMPHXXX{not} to be unde...,0.08,0.47,0.60,+0.52
deprel_nummod,Numeric modifier,"..., make his XXXEMPHXXX{one}-shot selec...",-0.26,0.14,0.23,+0.50
deprel_appos,Appositional modifier,...whether XXXEMPHXXX{workers} or consum...,-0.28,0.04,0.15,+0.43
deprel_aux:pass,Passive auxiliary,...ian Dasein XXXEMPHXXX{was} absorbed i...,-0.11,0.37,0.32,+0.43
deprel_ccomp,Clausal complement,"...egin with, XXXEMPHXXX{go} back to the...",0.13,0.52,0.55,+0.42
deprel_mark,Marker,...ecessary XXXEMPHXXX{because} it lies...,0.48,0.80,0.87,+0.39
pos_MD,Modal,...of ours XXXEMPHXXX{should} be exchang...,0.23,0.43,0.62,+0.39


In [13]:
out=df_to_latex_table(xdf, caption="""
Top 25 most Features whose predictive weight for philosophy rises most between the 1925--1950 and 2000--2025 classifiers. Columns show the logistic regression coefficient in each period and the difference between them. Positive values indicate association with philosophy; negative values indicate association with the comparator.
""".strip(), label="table:feat_diff_phil_rising", size="\\footnotesize")
out = out.replace("XXXEMPHXXX\{","\\textbf{")
out = out.replace("\\}","}")
out = out.replace("textbf{\{}Ln}", "textbf{\{}Ln\\textbf{\}}")
out = out.replace("Δ","$\Delta$")
with open('../../../Dropbox/Prof/Articles/OSP/tables/feat_diff_phil_rising2.tex','w') as f:
    f.write(out)

print(out)

\begin{table}[H]
  \centering
  \footnotesize
  \begin{tabular}{llllll}
  \toprule
  Feature & Example & 1900-1950 & 1950-2000 & 2000-2025 & $\Delta$(C21--eC20) \\
  \midrule
  Symbol & ...seen shape \textbf{=} rectangulari... & -0.40 & 0.12 & 0.26 & +0.67 \\
  Verb, base form & ...vation to \textbf{imply} that this... & -0.05 & 0.40 & 0.57 & +0.62 \\
  Adverbial modifier & ...ndition is \textbf{not} to be unde... & 0.08 & 0.47 & 0.60 & +0.52 \\
  Numeric modifier & ..., make his \textbf{one}-shot selec... & -0.26 & 0.14 & 0.23 & +0.50 \\
  Appositional modifier & ...whether \textbf{workers} or consum... & -0.28 & 0.04 & 0.15 & +0.43 \\
  Passive auxiliary & ...ian Dasein \textbf{was} absorbed i... & -0.11 & 0.37 & 0.32 & +0.43 \\
  Clausal complement & ...egin with, \textbf{go} back to the... & 0.13 & 0.52 & 0.55 & +0.42 \\
  Marker & ...ecessary \textbf{because} it lies... & 0.48 & 0.80 & 0.87 & +0.39 \\
  Modal & ...of ours \textbf{should} be exchang... & 0.23 & 0.43 & 0.62 & +0.39 

In [6]:
# xdf = odf.copy().sort_values('feat_diff', ascending=True)
# xdf = xdf[xdf['1925-1950'] > 0.05]
# xdf = xdf[xdf['feat_diff'] < -0.05]
# xdf = xdf[['feat_desc', 'eg', '1925-1950', '2000-2025','feat_diff']]
# xdf = xdf.set_index(['feat_desc','eg']).applymap(lambda x: f'{"+" if x > 0 else ""}{x:.2f}')
# xdf = xdf.reset_index()
# xdf

In [7]:
# out=df_to_latex_table(xdf, caption="""
# Top 25 most distinctive features of philosophy articles. Averages reflect average frequency per 1,000 words. Z-scores express the number of standard deviations from the mean frequency across all articles in the corpus.
# """.strip())
# out = out.replace("XXXEMPHXXX\{","\\textbf{")
# out = out.replace("\\}","}")
# with open('../../../Dropbox/Prof/Articles/OSP/tables/feat_diff_phil_falling.tex','w') as f:
#     f.write(out)


In [8]:
# # top_feats = list(set(odf.sort_values('feat_diff', ascending=False).head(25).index.tolist() + odf.sort_values('feat_diff', ascending=True).head(25).index.tolist()))
# # top_feats += list(set(odf.sort_values('feat_score1', ascending=False).head(25).index.tolist() + odf.sort_values('feat_score2', ascending=False).head(25).index.tolist()))
# odf = odf[odf.index.isin(top_feats)]
# odf = odf.reset_index().sort_values('feat_diff', ascending=False)


In [9]:
# fig = (
#     p9.ggplot(odf, p9.aes(x='1925-1950', y='2000-2025',label='feat_desc'))
#     + p9.geom_point(p9.aes(color='feat_type'), size=1, position=p9.position_nudge(x=0, y=-0.02),shape='x')
#     + p9.geom_text(p9.aes(color='feat_type'), size=8)#, position=p9.position_jitter(width=0.01, height=0.01))
#     + p9.scale_x_continuous(limits=(-.7,.7))
#     + p9.scale_y_continuous(limits=(-.7,.7))
#     + p9.theme_minimal()
#     + p9.geom_abline(intercept=0, slope=1, linetype='dashed', color='gray')
#     + p9.scale_color_gray()
#     # + p9.facet_wrap('feat_type')
# )
# fig